In [0]:
import mlflow
from mlflow.models.signature import infer_signature
from mlflow.client import MlflowClient
from xgboost import XGBClassifier
import json, xgboost as xgb
from datetime import datetime
import pickle

In [0]:
experiment_name = '/Workspace/Shared/pe_memberdna/bbm_propensity_model'

mlflow.xgboost.autolog(disable=False, log_input_examples=True, log_models=False, log_datasets=False)
mlflow.sklearn.autolog(disable=False, log_input_examples=True, log_models=False, log_datasets=False)

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri('databricks-uc')

if mlflow.get_experiment_by_name(experiment_name) is None:
    mlflow.create_experiment(name=experiment_name) 
mlflow.set_experiment(experiment_name)

In [0]:
%pip install skops==0.10.0
dbutils.library.restartPython()

In [0]:
%run ../../config/utils

In [0]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
import joblib
from sklearn.preprocessing import OneHotEncoder
from dataframe_imputer import DataFrameImputer

pkl_path = f"/Volumes/{catalog_name}/pe/helpers/bbm_propensity_model/XGBoost_DataPreprocessing_Pipeline.pkl"

preproc = joblib.load(pkl_path)


# ---------- 1) Patch legacy SimpleImputer instances ----------
def _patch_simple_imputer(si: SimpleImputer):
    if not hasattr(si, "_fit_dtype"):
        st = getattr(si, "statistics_", None)
        si._fit_dtype = getattr(st, "dtype", np.dtype("float64"))
    if not hasattr(si, "keep_empty_features"):
        si.keep_empty_features = False


def _compute_n_features_outs(ohe: OneHotEncoder):
    # Build list of output widths per input categorical column.
    cats = getattr(ohe, "categories_", None)
    if cats is None:
        raise RuntimeError("OneHotEncoder is not fitted: missing categories_.")
    drop_idx = getattr(ohe, "drop_idx_", None)
    if drop_idx is None:
        drop_mask = [False] * len(cats)
    else:
        if isinstance(drop_idx, np.ndarray):
            drop_idx = drop_idx.tolist()
        elif not isinstance(drop_idx, list):
            # Extremely rare pickles could store a scalar; broadcast if so
            drop_idx = [drop_idx] * len(cats)
        drop_mask = [di is not None for di in drop_idx]
    return [len(c) - (1 if m else 0) for c, m in zip(cats, drop_mask)]


def _patch_one_hot_encoder(ohe: OneHotEncoder, cols=None):
    if not hasattr(ohe, "_infrequent_enabled"):
        ohe._infrequent_enabled = False
    if not hasattr(ohe, "_drop_idx_after_grouping"):
        ohe._drop_idx_after_grouping = None
    if not hasattr(ohe, "_n_features_outs"):
        ohe._n_features_outs = _compute_n_features_outs(ohe)
    if hasattr(ohe, "sparse") and not hasattr(ohe, "sparse_output"):
        # bridge old -> new flag
        try: ohe.sparse_output = ohe.sparse
        except Exception: pass
    if cols is not None and not hasattr(ohe, "feature_names_in_"):
        ohe.feature_names_in_ = np.array(cols, dtype=object)
    if not hasattr(ohe, "drop_idx_"):
        ohe.drop_idx_ = None
    # be tolerant to unseen categories
    try:
        ohe.set_params(handle_unknown="ignore")
    except Exception:
        pass


def patch_preproc(obj, cols_context=None):

    # ColumnTransformer
    if isinstance(obj, ColumnTransformer):
        for name, trans, cols in obj.transformers_:
            patch_preproc(trans, cols_context=cols)
        # also check remainder if it’s an estimator
        rem = getattr(obj, "remainder", None)
        if rem not in (None, "drop", "passthrough"):
            patch_preproc(rem, cols_context=None)
        return obj

    # Pipeline
    if isinstance(obj, Pipeline):
        for _, step in obj.steps:
            patch_preproc(step, cols_context=cols_context)
        return obj

    # Leaf estimators
    if isinstance(obj, SimpleImputer):
        _patch_simple_imputer(obj)
    if isinstance(obj, OneHotEncoder):
        _patch_one_hot_encoder(obj, cols=cols_context)
    return obj



# ---- call it on your loaded preproc ----
preproc = patch_preproc(preproc)




# ---------- 2) Extract column lists ----------
num_cols = (preproc.named_transformers_['num'].feature_names_in_.tolist()
            if hasattr(preproc.named_transformers_['num'], 'feature_names_in_')
            else preproc.transformers_[0][2])
cat_cols = preproc.transformers_[1][2]




# ---------- 3) Build fake data with correct dtypes ----------

# Numeric: floats; Categorical: pick a known category per column from fitted OHE
ohe = preproc.named_transformers_['cat'].named_steps['one_hot_encoding']
known_cat = {col: ohe.categories_[i][0] for i, col in enumerate(cat_cols)}  # guaranteed known

rows = []
for _ in range(5):
    row = {c: 0.0 for c in num_cols}     # numeric as float
    row.update(known_cat)                # categorical as strings
    rows.append(row)


test_df = pd.DataFrame(rows)[num_cols + cat_cols]

# ensure dtypes are what the pipeline expects
test_df[num_cols] = test_df[num_cols].astype("float64")
for c in cat_cols:
    test_df[c] = test_df[c].astype("object")




# ---------- 4) Transform ----------
Xt = preproc.transform(test_df)
print("Transformed shape:", getattr(Xt, "shape", None))


In [0]:
import numpy as np, pandas as pd, itertools
from scipy import sparse as sp
from sklearn.preprocessing import OneHotEncoder

# --- pull columns/cats from your fitted preproc ---
num_cols = (preproc.named_transformers_['num'].feature_names_in_.tolist()
            if hasattr(preproc.named_transformers_['num'], 'feature_names_in_')
            else preproc.transformers_[0][2])
cat_cols = preproc.transformers_[1][2]

ohe: OneHotEncoder = preproc.named_transformers_['cat'].named_steps['one_hot_encoding']
cat_values_per_col = [list(cats) for cats in ohe.categories_]

# Cartesian product of all categorical values (usually small; in your case ≈ 12 rows)
cat_product_rows = list(itertools.product(*cat_values_per_col))

# Create varied numeric values (repeat for each categorical combo)
rng = np.random.default_rng(42)
def make_numeric_row():
    # deterministic but varied
    return {c: float(rng.normal(loc=0.0, scale=1.0)) for c in num_cols}

rows = []
for combo in cat_product_rows:
    r = make_numeric_row()
    r.update({c: v for c, v in zip(cat_cols, combo)})
    rows.append(r)

test_df = pd.DataFrame(rows)[num_cols + cat_cols]
# dtypes the pipeline expects
test_df[num_cols] = test_df[num_cols].astype("float64")
for c in cat_cols:
    test_df[c] = test_df[c].astype("object")

print(test_df.head(10))
print("Test rows:", len(test_df), " | num_cols:", len(num_cols), " | cat_cols:", len(cat_cols))


In [0]:
test_df

In [0]:
# Transform
Xt = preproc.transform(test_df)
if sp.issparse(Xt):
    Xt = Xt.toarray()

# Build feature names for ColumnTransformer output
def ct_feature_names(ct) -> list[str]:
    names = []
    for name, trans, cols in ct.transformers_:
        # unwrap pipelines
        final_est = trans
        if hasattr(trans, "steps"):
            final_est = trans.steps[-1][1]  # last step

        if isinstance(final_est, OneHotEncoder):
            # use OHE names
            # prefer get_feature_names_out if available
            if hasattr(final_est, "get_feature_names_out") and False:
                names.extend(list(final_est.get_feature_names_out(cols)))
            else:
                # fallback for very old OHEs
                for c, cats in zip(cols, final_est.categories_):
                    names.extend([f"{c}_{str(v)}" for v in cats])
        else:
            # numeric block keeps original column names
            names.extend(list(cols))
    return names

out_cols = ct_feature_names(preproc)
df_out = pd.DataFrame(Xt, columns=out_cols)

print(df_out.shape)
df_out.head()


In [0]:
import mlflow, sklearn
from mlflow.models import infer_signature
from datetime import datetime

registered_model_name = f"{catalog_name}.pe.bbm_preprocessor"  

mark_datetime = datetime.strftime(datetime.now(), '_%Y-%m-%d_%H-%M-%S')

with mlflow.start_run(run_name=f'importing_bbm_preprocessing_{mark_datetime}') as run:

    # small input/output samples for signature
    input_example = test_df.head(5)
    output_example = preproc.transform(input_example)

    signature = infer_signature(input_example, output_example)

    # lock the env that worked for you

    # If you have a local module file for DataFrameImputer, add its path here (Databricks workspace path or local file path)
    custom_code_paths = ['./dataframe_imputer.py']  # e.g., ["/Workspace/Users/you@company.com/path/to/dataframe_imputer.py"]

 
    model_info = mlflow.sklearn.log_model(
        sk_model=preproc,
        artifact_path="preprocessor",
        signature=signature,
        input_example=input_example,
        registered_model_name=registered_model_name,
        code_paths=custom_code_paths,   # keep empty if no custom class needed at inference
    )


In [0]:
client = MlflowClient()
client.set_registered_model_alias(f'{catalog_name}.pe.bbm_preprocessor', 'champion', model_info.registered_model_version)

In [0]:
mark_datetime = datetime.strftime(datetime.now(), '_%Y-%m-%d_%H-%M-%S')

with mlflow.start_run(run_name=f'importing_bbm_model_{mark_datetime}') as run:
    clf = xgb.XGBClassifier()
    clf.load_model(f"/Volumes/{catalog_name}/pe/helpers/bbm_propensity_model/XGBoost_Model.json")

    mlflow.log_artifact(f'/Volumes/{catalog_name}/pe/helpers/bbm_propensity_model/XGBoost_Model.meta.json')

    example_in = Xt[:5]
    example_out = clf.predict_proba(example_in)[:, 1]
    signature = infer_signature(Xt, example_out)

    model_info = mlflow.xgboost.log_model(
        xgb_model=clf,
        artifact_path="model",
        signature=signature,
        input_example=example_in,
        registered_model_name=f"{catalog_name}.pe.bbm_model"  # UC path if desired
    )

In [0]:
client = MlflowClient()
client.set_registered_model_alias(f'{catalog_name}.pe.bbm_model', 'champion', model_info.registered_model_version)